# Stage 1 — Obtain the data

This notebook downloads every series used in the project from the U.S. Energy Information
Administration (EIA), saves each one as a CSV named after the metric it measures, and
writes `data/raw/manifest.csv`, which records the coverage of each file.

The two kinds of data come from **two different EIA products**:

| Data | Source | Why |
|---|---|---|
| 11 weekly petroleum series | `.xls` history files | complete, no key needed |
| 2 daily spot prices (WTI, Brent) | **EIA API v2** | the published `.xls` price files silently skip hundreds of ordinary trading days |

| Folder | Content | In Git? |
|---|---|---|
| `data/raw/xls/` | original EIA `.xls` downloads | no |
| `data/raw/weekly/` | weekly petroleum data as CSV, e.g. `crude_stocks.csv` (public domain) | yes |
| `data/raw/prices/` | daily WTI and Brent prices as CSV (third-party data, see README) | no |

Files that are already present are not downloaded again, so the notebook can be re-run
quickly.

**The API needs a free key.** It is read from `EIA_API_KEY` in a file called `.env` in the
repository root, which is in `.gitignore`. The key is never printed by this notebook. See
the README for how to create it.

*Written with the help of Claude (Anthropic); declared in the paper appendix.*

In [1]:
import sys
from pathlib import Path

# Find the repository root (the folder containing src/) so `src` can be imported
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import MANIFEST_PATH
from src.data_utils import (api_route_metadata, api_series_names, build_manifest,
                            compare_price_sources, convert_xls_to_csv, download_all,
                            download_prices, observations_per_year, read_series,
                            series_table, weekly_observations_per_year)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("Repository root:", ROOT)

Repository root: /Users/yazidabaroudi/Desktop/ADA project


## 1. The series used

In [2]:
series_table()

,series_id,metric,description,units,frequency,csv_file,url
0,RWTC,wti,"WTI spot price, Cushing OK (FOB)",USD per barrel,daily,data/raw/prices/wti.csv,https://www.eia.gov/dnav/pet/hist_xls/RWTCd.xls
1,RBRTE,brent,"Brent spot price, Europe (FOB)",USD per barrel,daily,data/raw/prices/brent.csv,https://www.eia.gov/dnav/pet/hist_xls/RBRTEd.xls
2,WCESTUS1,crude_stocks,"U.S. commercial crude oil stocks, excl. SPR",thousand barrels,weekly,data/raw/weekly/crude_stocks.csv,https://www.eia.gov/dnav/pet/hist_xls/WCESTUS1w.xls
3,W_EPC0_SAX_YCUOK_MBBL,cushing_stocks,Cushing OK crude oil stocks,thousand barrels,weekly,data/raw/weekly/cushing_stocks.csv,https://www.eia.gov/dnav/pet/hist_xls/W_EPC0_SAX_YCUOK_MBBLw.xls
4,WCSSTUS1,spr_stocks,Strategic Petroleum Reserve crude oil stocks,thousand barrels,weekly,data/raw/weekly/spr_stocks.csv,https://www.eia.gov/dnav/pet/hist_xls/WCSSTUS1w.xls
5,WGTSTUS1,gasoline_stocks,U.S. total motor gasoline stocks,thousand barrels,weekly,data/raw/weekly/gasoline_stocks.csv,https://www.eia.gov/dnav/pet/hist_xls/WGTSTUS1w.xls
6,WDISTUS1,distillate_stocks,U.S. distillate fuel oil stocks,thousand barrels,weekly,data/raw/weekly/distillate_stocks.csv,https://www.eia.gov/dnav/pet/hist_xls/WDISTUS1w.xls
7,WCRFPUS2,crude_production,U.S. field production of crude oil,thousand barrels per day,weekly,data/raw/weekly/crude_production.csv,https://www.eia.gov/dnav/pet/hist_xls/WCRFPUS2w.xls
8,WCRRIUS2,refinery_input,U.S. refiner net input of crude oil,thousand barrels per day,weekly,data/raw/weekly/refinery_input.csv,https://www.eia.gov/dnav/pet/hist_xls/WCRRIUS2w.xls
9,WPULEUS3,refinery_utilization,U.S. refinery utilisation (% of operable capacity),percent,weekly,data/raw/weekly/refinery_utilization.csv,https://www.eia.gov/dnav/pet/hist_xls/WPULEUS3w.xls


## 2. Check the API key and confirm the series codes

The request below is the smallest one the API offers: the metadata of the spot price
route. It confirms three things — that the key in `.env` is valid, that the route has a
`daily` frequency, and that the facet used to pick a series is called `series`.

Nothing here prints the key. If this cell fails with a message about the key, create or
correct `.env` as described in the README.

In [3]:
meta = api_route_metadata()
print("route period range:", meta["startPeriod"], "to", meta["endPeriod"])
print("frequencies available:", [f["id"] for f in meta["frequency"]])
print("facets available:", [f["id"] for f in meta["facets"]])
print("\nThe API key works.")

route period range: 1986-01-03 to 2026-10-30
frequencies available: ['monthly', 'weekly', 'annual', 'daily']
facets available: ['duoarea', 'product', 'process', 'series']

The API key works.


In [4]:
# The two codes the project needs, as the API itself describes them
names = api_series_names()
names[names["series_id"].isin(["RWTC", "RBRTE"])]

,series_id,description
6,RBRTE,Europe Brent Spot Price FOB (Dollars per Barrel)
7,RWTC,"Cushing, OK WTI Spot Price FOB (Dollars per Barrel)"


## 3. Download the original EIA `.xls` files

Each file is about 100–500 KB. If a download fails, the reason is shown in the `status`
column and the other files are still downloaded. These files provide the 11 weekly series;
the two price files are downloaded as well, but only to check the API against them.

In [5]:
status = download_all()
status

,series_id,status,file
0,RWTC,already present,RWTCd.xls
1,RBRTE,already present,RBRTEd.xls
2,WCESTUS1,already present,WCESTUS1w.xls
3,W_EPC0_SAX_YCUOK_MBBL,already present,W_EPC0_SAX_YCUOK_MBBLw.xls
4,WCSSTUS1,already present,WCSSTUS1w.xls
5,WGTSTUS1,already present,WGTSTUS1w.xls
6,WDISTUS1,already present,WDISTUS1w.xls
7,WCRFPUS2,already present,WCRFPUS2w.xls
8,WCRRIUS2,already present,WCRRIUS2w.xls
9,WPULEUS3,already present,WPULEUS3w.xls


## 4. Convert the weekly `.xls` files to CSV

Each `.xls` file is recognised from the EIA code stored inside it and saved as
`date, <metric>`. The two price files are deliberately skipped here — they are replaced by
the API download in the next step.

In [6]:
converted = convert_xls_to_csv()
converted

,file,series_id,saved_as,status
0,RBRTEd.xls,RBRTE,NaN,skipped: written by download_prices() from the EIA API
1,RWTCd.xls,RWTC,NaN,skipped: written by download_prices() from the EIA API
2,WCEIMUS2w.xls,WCEIMUS2,data/raw/weekly/crude_imports.csv,converted
3,WCESTUS1w.xls,WCESTUS1,data/raw/weekly/crude_stocks.csv,converted
4,WCREXUS2w.xls,WCREXUS2,data/raw/weekly/crude_exports.csv,converted
5,WCRFPUS2w.xls,WCRFPUS2,data/raw/weekly/crude_production.csv,converted
6,WCRRIUS2w.xls,WCRRIUS2,data/raw/weekly/refinery_input.csv,converted
7,WCSSTUS1w.xls,WCSSTUS1,data/raw/weekly/spr_stocks.csv,converted
8,WDISTUS1w.xls,WDISTUS1,data/raw/weekly/distillate_stocks.csv,converted
9,WGTSTUS1w.xls,WGTSTUS1,data/raw/weekly/gasoline_stocks.csv,converted


## 5. Download the daily prices from the EIA API

The API returns at most 5000 rows per request, so each series is fetched in pages with an
increasing `offset`, sorted by date ascending.

The published `.xls` price files are still used for two checks: the prices must be
identical on every date both sources cover, and the few days that are present only in the
`.xls` are added back, so that neither source's gaps survive.

In [7]:
prices_report = download_prices()
prices_report

,metric,series_id,n_api,n_xls,n_differences,added_by_api,filled_from_xls,n_saved,saved_as
0,wti,RWTC,10286,9537,0,757,8,10294,data/raw/prices/wti.csv
1,brent,RBRTE,10013,9122,3,900,9,10022,data/raw/prices/brent.csv


## 6. Checks

### 6.1 Daily prices per year

A full year of trading has about 250 days. Any year below 240 is listed: only the first
year of each series (partial history) and the current year (incomplete) should appear.

In [8]:
counts = pd.concat([observations_per_year("wti"), observations_per_year("brent")], axis=1)
print("daily prices per year:")
display(counts.T)

low = counts[(counts < 240).any(axis=1)]
print("\nyears with fewer than 240 observations:")
display(low if not low.empty else "none")

daily prices per year:


year,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
wti,251.0,254.0,257.0,257.0,257.0,256.0,257.0,250.0,252.0,251.0,...,250.0,249.0,256.0,252.0,251.0,251.0,248.0,252.0,248.0,213.0
brent,NaN,160.0,255.0,254.0,256.0,257.0,257.0,252.0,252.0,253.0,...,256.0,254.0,259.0,255.0,253.0,252.0,251.0,255.0,253.0,213.0



years with fewer than 240 observations:


,wti,brent
year,,
1987,254,160.0
2026,213,213.0


### 6.2 The API against the published `.xls` files

`price_differs` counts the dates where the two sources give a different price; it should be
0, or very close to it. `added_by_api` counts the trading days the `.xls` file was missing,
and `only_in_xls` the few days the API is missing, which are filled from the `.xls`.

In [9]:
for name in ["wti", "brent"]:
    table = compare_price_sources(name)
    if table.empty:
        print(f"{name}: the .xls file is not present, so no comparison is possible\n")
        continue
    totals = table[["added_by_api", "only_in_xls", "price_differs"]].sum()
    print(f"=== {name.upper()} ===")
    print(f"days added by the API      : {totals['added_by_api']}")
    print(f"days only in the .xls file : {totals['only_in_xls']} (filled back in)")
    print(f"dates where prices differ  : {totals['price_differs']}")
    worst = table[table["added_by_api"] > 0].nlargest(5, "added_by_api")
    print("the five years the .xls file was missing most days:")
    display(worst)

=== WTI ===
days added by the API      : 757
days only in the .xls file : 8 (filled back in)
dates where prices differ  : 0
the five years the .xls file was missing most days:


,n_api,n_xls,added_by_api,only_in_xls,price_differs
year,,,,,
2003,250,139,111,0,0
1999,251,165,86,0,0
2004,249,185,64,0,0
1996,254,198,56,0,0
1997,252,203,49,0,0


=== BRENT ===
days added by the API      : 900
days only in the .xls file : 9 (filled back in)
dates where prices differ  : 3
the five years the .xls file was missing most days:


,n_api,n_xls,added_by_api,only_in_xls,price_differs
year,,,,,
1995,253,127,126,0,0
1994,252,138,114,0,0
1999,249,173,76,0,0
1996,254,183,71,0,0
1998,253,189,64,0,0


### 6.3 The weekly EIA series

Each weekly series should have 52 or 53 observations per year. Years with fewer are listed:
only the first year of each series, the current year, and the early 1980s (when EIA
published less often) should appear.

In [10]:
weekly_counts = weekly_observations_per_year()
gaps = weekly_counts[(weekly_counts < 52).any(axis=1)]
print("years where a weekly series has fewer than 52 observations:")
display(gaps.fillna(0).astype(int))

in_sample = weekly_counts.loc[2000:2025]
odd = in_sample[(in_sample.notna() & (in_sample < 52)).any(axis=1)]
print("\nsame check restricted to 2000-2025 (the project sample, excluding 2026):")
display(odd.fillna(0).astype(int) if not odd.empty else "none - every series has 52-53 weeks every year")

years where a weekly series has fewer than 52 observations:


,crude_stocks,cushing_stocks,spr_stocks,gasoline_stocks,distillate_stocks,crude_production,refinery_input,refinery_utilization,crude_imports,crude_exports,product_supplied
year,,,,,,,,,,,
1982,17,0,17,0,17,0,17,0,17,0,0
1983,49,0,49,0,49,49,49,0,49,0,0
1990,52,0,52,52,52,52,52,9,52,0,8
1991,52,0,52,52,52,52,52,52,52,47,51
2004,53,39,53,53,53,53,53,53,53,53,53
2026,44,44,44,44,44,44,44,44,44,44,44



same check restricted to 2000-2025 (the project sample, excluding 2026):


,crude_stocks,cushing_stocks,spr_stocks,gasoline_stocks,distillate_stocks,crude_production,refinery_input,refinery_utilization,crude_imports,crude_exports,product_supplied
year,,,,,,,,,,,
2004,53,39,53,53,53,53,53,53,53,53,53


## 7. Check the files and write the manifest

In [11]:
manifest = build_manifest()
manifest[["metric", "frequency", "source", "status", "first_date", "last_date", "n_obs", "n_missing"]]

,metric,frequency,source,status,first_date,last_date,n_obs,n_missing
0,wti,daily,EIA API v2,ok,1986-01-02,2026-11-03,10294,0
1,brent,daily,EIA API v2,ok,1987-05-20,2026-11-03,10022,0
2,crude_stocks,weekly,EIA .xls history file,ok,1982-08-20,2026-10-30,2301,0
3,cushing_stocks,weekly,EIA .xls history file,ok,2004-04-09,2026-10-30,1178,0
4,spr_stocks,weekly,EIA .xls history file,ok,1982-08-20,2026-10-30,2301,0
5,gasoline_stocks,weekly,EIA .xls history file,ok,1990-01-05,2026-10-30,1922,0
6,distillate_stocks,weekly,EIA .xls history file,ok,1982-08-20,2026-10-30,2301,0
7,crude_production,weekly,EIA .xls history file,ok,1983-01-07,2026-10-30,2284,0
8,refinery_input,weekly,EIA .xls history file,ok,1982-08-20,2026-10-30,2301,0
9,refinery_utilization,weekly,EIA .xls history file,ok,1990-11-02,2026-10-30,1879,0


In [12]:
problems = manifest[manifest["status"] != "ok"]
if not problems.empty:
    raise RuntimeError(
        "These series are missing or unreadable:\n"
        + problems[["metric", "status", "url"]].to_string(index=False)
    )
print(f"All {len(manifest)} metrics are available as CSV.")
print("Manifest saved to", MANIFEST_PATH.relative_to(ROOT))

All 13 metrics are available as CSV.
Manifest saved to data/raw/manifest.csv


## 8. Sanity check

The WTI spot price was negative on 20 April 2020. Seeing that value here confirms the
price series was downloaded correctly; it is dealt with in stage 2.

In [13]:
wti = read_series("wti")
wti.loc["2020-04-17":"2020-04-22"]

date
2020-04-17    18.31
2020-04-20   -36.98
2020-04-21     8.91
2020-04-22    13.64
Name: wti, dtype: float64

**Next:** `02_clean_data.ipynb` turns these files into one weekly table aligned to EIA
release dates.